# Agentic AI Workshop — From Theory to Code

This notebook turns every concept from the workshop's theory segment into runnable code: agentic loops, schema-enforced control flow, tool calling, retrieval-augmented generation, and MCP.

**Design rule for the whole notebook:** every LLM call goes through the one OpenAI-compatible client built in Section 0. The default backend here is a small (1.5B parameter), CPU-only, non-gated local model — chosen so this runs on a free Google Colab instance with no GPU, no API key, and no license click-through. Swapping to Ollama, vLLM, or a bigger model on real hardware later only ever means changing `BASE_URL` (and maybe `MODEL`) in that one cell — nothing in Sections 1-6 talks to a backend directly.

**Contents**
0. Model access layer — a local OpenAI-compatible server + `chat()`
1. What is Agentic AI? — single-shot vs. a loop
2. Control Loops and Schema — CoT, ReAct, schema-enforced termination
3. Tool Calling — function definitions, the orchestrator, a worked example
4. RAG — chunk/embed/retrieve, then a small corrective loop
5. MCP — a real FastMCP server + client, wired into the Section 3 orchestrator
6. Bringing it together — one closing mini-agent


In [ ]:
# Colab: if you're opening this notebook fresh (not already inside a clone of the repo),
# run this cell to clone it. Safe to re-run -- skips cleanly if already cloned.
import os
if not os.path.exists("ai-for-all-mississippi-agentic-ai") and "llama_server_utils.py" not in os.listdir("."):
    !git clone https://github.com/kuiper69/ai-for-all-mississippi-agentic-ai.git

In [ ]:
# Colab quirk: opening a notebook from a cloned repo does NOT change the working
# directory to the repo folder -- it stays at /content, so `import llama_server_utils`
# and the model/log paths below would fail. This finds the repo directory (wherever it
# was cloned) and cd's into it. Safe to re-run; does nothing outside Colab if you already
# launched Jupyter from inside the repo folder.
import os, glob

if "llama_server_utils.py" not in os.listdir("."):
    matches = glob.glob("/content/**/llama_server_utils.py", recursive=True)
    if not matches:
        matches = glob.glob("**/llama_server_utils.py", recursive=True)
    if matches:
        os.chdir(os.path.dirname(os.path.abspath(matches[0])))
    else:
        raise FileNotFoundError(
            "Could not find llama_server_utils.py. If you're not in Colab, "
            "cd into the cloned repo directory before launching Jupyter."
        )

print("Working directory:", os.getcwd())

## Section 0 — Model Access Layer

Ollama, vLLM, LM Studio, and llama.cpp (the local-hosting stack from the slides) all speak the same OpenAI-compatible `/v1/chat/completions` endpoint. We stand up that endpoint locally with **llama-cpp-python's built-in server** — a pure `pip install`, no system binary or install script, which is what makes this Colab-friendly — serving **Qwen2.5-1.5B-Instruct** (Apache-2.0, fully open, ~1GB at Q4 quantization, downloaded once from Hugging Face with no login required).

Every later section calls one function, `chat()`, never the raw client — so pointing this at Ollama on a real GPU box later is a one-line change, not a rewrite.

In [ ]:
# Run once per environment (Colab: run this cell, it's fast -- these are prebuilt CPU wheels, not a source build):
# %pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu "llama-cpp-python[server]" openai pydantic huggingface_hub

In [ ]:
from llama_server_utils import start_llama_server, stop_llama_server

# Downloads the model on first run (~1GB, cached after that) and starts the server
# in the background. Takes ~10-30s once the model is on disk.
start_llama_server()

In [ ]:
from openai import OpenAI

# --- Point this at whichever backend is currently running -----------------
# llama.cpp (started above):
#   BASE_URL = "http://127.0.0.1:8000/v1"
# Ollama (needs the /v1 suffix explicitly):
#   BASE_URL = "http://localhost:11434/v1"
# vLLM (OpenAI-compatible server mode):
#   BASE_URL = "http://localhost:8000/v1"
#
# Swap only these lines to change backends -- nothing below this cell changes.
BASE_URL = "http://127.0.0.1:8000/v1"
API_KEY = "not-needed"   # local backends ignore this, but the client requires a non-empty string
MODEL = "qwen2.5-1.5b"   # llama-cpp-python serves whatever --model it was started with; this label is just for logging

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

In [ ]:
def chat(messages, tools=None, tool_choice=None, temperature=0.2, **kwargs):
    """
    Thin wrapper around client.chat.completions.create().
    Every section below calls this function only -- never `client` directly --
    so the backend stays swappable from the config cell above.

    Gotcha (verified against llama-cpp-python's server): passing `tools` without
    an explicit `tool_choice` gets silently ignored -- the model answers in plain
    text instead of calling anything. Pass tool_choice="auto" to actually enable
    tool use. This is backend-specific behavior worth re-checking if you swap to
    Ollama or vLLM later.

    Returns an OpenAI `message` object (has .content and .tool_calls).
    """
    kwargs = dict(model=MODEL, messages=messages, temperature=temperature, **kwargs)
    if tools is not None:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice or "auto"
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message

**Smoke test** — confirms the server is up and `chat()` works. If this cell errors, check `llama_server.log` (written next to the notebook) before moving on.

In [ ]:
reply = chat([{"role": "user", "content": "In one short sentence, what is an AI agent?"}])
print(reply.content)

## Section 1 — What is Agentic AI?

**Formal definition** (Xi et al., "The Rise and Potential of Large Language Model Based Agents: A Survey," arXiv 2023): *"An artificial entity capable of perceiving its surroundings using sensors, making decisions, and then taking actions in response using actuators."*

**In plain terms:** an LLM can only decide based on two things — what it learned during training, and whatever is in the prompt in front of it right now. It has no sensors and no actuators of its own. Agentic *software* is what builds the sensors (fetching real data into the prompt) and the actuators (mapping the model's output onto code that actually runs) around the model. A chatbot is one module inside that picture, not the whole system.

**One-liner worth keeping:** most of what an agent *does* is autonomously execute existing software — the model decides *which* software and *with what arguments*, but the actual work (an API call, a file write, a database query) is regular code, same as before agents existed.

**GAIA benchmark** (Mialon et al., "GAIA: a benchmark for general AI assistants," arXiv:2311.12983) gives a concrete way to talk about "how agentic" a task is, by the number of tools and steps it needs:

| Level | Definition | Human accuracy |
|---|---|---|
| 1 | No tools, or at most one tool, ≤5 steps | 94% |
| 2 | ~5-10 steps, combining different tools | 92% |
| 3 | Arbitrarily long action sequences, any number of tools | 87% |

Humans stay around 90%+ across all three levels; the paper's headline result is GPT-4 with no tools scoring 15% on the full set. The gap isn't model intelligence — it's the missing loop. Which is exactly what the rest of this notebook builds, one piece at a time.

### Single-shot vs. a loop

The cleanest way to see the gap: ask the model something it cannot possibly know from training data alone, with no help. It only has two inputs — training knowledge and the prompt — so today's date isn't in either.

In [ ]:
reply = chat([{"role": "user", "content": "What is today's date? Just answer directly."}])
print("Single-shot answer:", reply.content)

Whatever it just said, it's either a hedge ("I don't have real-time access") or a guess — it has no way to be right. Now the smallest possible loop: perceive the gap, decide on an action, act, then hand the *result* back to the model instead of asking it to guess.

This is a preview, not the real thing — the action here is hardcoded (`date.today()`), not something the model chose from a menu of options. Section 3 turns "act" into a general mechanism where the model picks the function and the arguments itself. But the shape — perceive → decide → act → use the result — is exactly the Agent Core loop from the overview slide, and it's worth seeing work end to end before adding that generality.

In [ ]:
from datetime import date

def think_decide_act(user_request):
    print("THINK:  this needs current information the model can't have on its own.")
    print("DECIDE: fetch the real date instead of asking the model to guess.")
    real_date = date.today().isoformat()
    print(f"ACT:    fetched {real_date}")
    grounded = chat([
        {"role": "user", "content": f"Today's date is {real_date}. {user_request}"}
    ])
    return grounded.content

print("\nLooped answer:", think_decide_act("What is today's date? Just answer directly."))

## Section 2 — Control Loops and Schema

**Chain-of-Thought** (Wei et al., "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models," arXiv:2201.11903): asking a model to reason step by step, rather than jump straight to an answer, measurably improves accuracy on multi-step problems. Below is the same word problem asked two ways.

In [ ]:
problem = ("A library has 4 shelves. Each shelf holds 12 books. If 15 books are checked out, "
           "how many books remain on the shelves?")

plain = chat([{"role": "user", "content": problem + " Answer with just the final number."}])
print("Plain answer:   ", plain.content)

cot = chat([{"role": "user", "content": problem + " Think step by step, then give the final number."}])
print("\nChain-of-thought:\n", cot.content)

(Correct answer: 4 x 12 = 48, 48 - 15 = 33.) Whatever the plain answer just was, watch whether it's actually 33 -- small models often botch the multiplication-then-subtraction chain when asked to skip straight to a number, and get there reliably once forced to show the intermediate step.

But the deck's own point about CoT is sharper than "it helps": reasoning *by itself* only creates an illusion of agency. Nothing outside the model changed between those two calls -- same weights, same tools (none), same lack of any real action. It's still a single forward pass producing text. An actual control loop needs to actually *do* something between reasoning steps: call a tool, observe a result, decide what's next. That's **ReAct** (Yao et al., "ReAct: Synergizing Reasoning and Acting in Language Models," ICLR 2023, arXiv:2210.03629) -- interleaved Thought / Action / Observation, repeated until done.

The open question ReAct's own error analysis raises: what stops the loop? Nothing, by default -- the paper's error analysis found the model repeating the same thought/action pair rather than terminating. A **schema** is what gives a loop a deterministic exit condition: an explicit `done` field the model must set, rather than us guessing from its prose whether it's finished.

We already know `response_format={"type": "json_object"}` plus a Pydantic model works reliably against this backend (verified in Section 0) -- so that's what drives the loop below, not a "reasoning until it sounds finished" heuristic.

In [ ]:
import json, ast, operator as op
from pydantic import BaseModel, Field

class AgentStep(BaseModel):
    thought: str = Field(..., description="internal reasoning about what to do next")
    action: str = Field(..., description="one of: calculate, finish")
    action_input: str = Field(..., description="a simple two-operand expression like '12 + 8' if action=calculate, or the final numeric answer if action=finish")
    done: bool = Field(..., description="true only when action is finish")

# A restricted evaluator -- never hand a model's raw string straight to eval().
_ops = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv, ast.USub: op.neg}
def safe_calc(expr):
    def _eval(node):
        if isinstance(node, ast.Constant):
            return node.value
        if isinstance(node, ast.BinOp):
            return _ops[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):
            return _ops[type(node.op)](_eval(node.operand))
        raise ValueError("unsupported expression")
    return _eval(ast.parse(expr, mode="eval").body)

schema = AgentStep.model_json_schema()
sys_prompt = (
    "You are a research agent that solves arithmetic problems using a calculator tool, one simple "
    "two-operand step at a time (e.g. '12 + 8', not the whole expression at once). Always use the "
    "most recent Observation value in your next step -- never repeat a calculation you already have "
    "the result for. Respond with ONLY a JSON object matching this schema, no other text: " + json.dumps(schema)
)

def run_react(question, max_steps=5):
    messages = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": question}]
    for step in range(max_steps):
        msg = chat(messages, response_format={"type": "json_object"}, max_tokens=200)
        parsed = AgentStep.model_validate_json(msg.content)
        print(f"[step {step}] thought={parsed.thought!r}")
        print(f"          action={parsed.action}  input={parsed.action_input!r}  done={parsed.done}")
        # Trust the *action*, not the "done" flag, to decide whether to execute --
        # small models sometimes set done=true on the same step as a real action.
        if parsed.action == "calculate":
            try:
                result = safe_calc(parsed.action_input)
            except Exception as e:
                result = f"error: {e}"
            print(f"          -> Observation: {result}")
            messages.append({"role": "assistant", "content": msg.content})
            messages.append({"role": "user", "content": f"Observation: {result}"})
        if parsed.action == "finish":
            return parsed.action_input
    return "(max steps reached without finishing)"

print("FINAL ANSWER:", run_react("What is (12 + 8) * 3? Break it into two calculator steps."))

If you re-run the cell above a few times, watch for a small model quirk: it can set `action: "calculate"` and `done: true` on the very same step (grammatically valid against the schema, logically contradictory). The schema guarantees the *shape* of every step; it says nothing about whether the reasoning inside that shape is sound -- which is exactly the gap Schema-Guided Dialogue (Rastogi et al., 2020) and Efficient Guided Generation (Willard & Louf, 2023) are about: a schema is a learned or enforced *convention*, not a correctness guarantee. That's why the loop above checks `action`, not `done`, before deciding whether to actually execute a step.

## Section 3 — Tool Calling

MCP's "Tools" primitive (and the OpenAI-style `tools` parameter we're about to use) is model-controlled: given a list of function signatures, the model decides *whether* to call one, *which* one, and *what arguments* to pass. The actual execution is ordinary code -- an **orchestrator** sits between "the model asked for a call" and "the call happened," since models never touch the network or the filesystem directly.

First, the mechanics working exactly as advertised — a single call, one tool, `tool_choice="auto"`:

In [ ]:
def get_weather(location: str) -> str:
    """Pretend weather lookup -- swap this for a real API call."""
    return f"18C, partly cloudy in {location}"

WEATHER_TOOL = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current weather for a location",
        "parameters": {"type": "object", "properties": {"location": {"type": "string"}}, "required": ["location"]},
    },
}]

msg = chat([{"role": "user", "content": "What's the weather in Milan?"}], tools=WEATHER_TOOL)
print("tool_calls:", msg.tool_calls)

That part is reliable: correct tool, correct arguments, every time we tried it. The unreliable part, verified against this exact backend and model size, is what happens *next* -- feeding the tool's result back and asking for a natural-language answer. In testing, this 1.5B model would confidently restate a plausible-looking but **wrong** number instead of the one the tool actually returned (e.g. inventing "48.97 USD" when the tool said 54.5), and with `tools` still attached it would sometimes just call the same tool again instead of ever answering. This is the Native-vs-Prompted distinction from the slides made concrete: native tool-calling's *first hop* works, but small local models cannot be trusted to faithfully restate a result in prose across a second hop.

The fix follows directly from Section 2: use the same schema-enforced loop for tool routing, and -- this is the important part -- **never ask the model to restate a number it already computed**. Once the orchestrator has every observation it needs, it builds the final answer itself, deterministically, in Python.

In [ ]:
from typing import Optional

def convert_currency(amount, from_currency, to_currency):
    """Fixed-rate mock -- swap this for a real FX API call."""
    rates = {("EUR", "USD"): 1.09, ("USD", "EUR"): 0.92}
    rate = rates.get((from_currency.upper(), to_currency.upper()))
    if rate is None:
        return f"error: no rate for {from_currency}->{to_currency}"
    return f"{amount} {from_currency} = {round(amount * rate, 2)} {to_currency}"

TOOL_FUNCS = {
    "get_weather": lambda **kw: get_weather(kw["location"]),
    "convert_currency": lambda **kw: convert_currency(kw["amount"], kw["from_currency"], kw["to_currency"]),
}

class ToolStep(BaseModel):
    thought: str = Field(..., description="internal reasoning about what to do next")
    tool_name: Optional[str] = Field(None, description="one of: get_weather, convert_currency -- or null once every part of the request is answered")
    location: Optional[str] = Field(None, description="required if tool_name=get_weather")
    amount: Optional[float] = Field(None, description="required if tool_name=convert_currency")
    from_currency: Optional[str] = Field(None, description="required if tool_name=convert_currency")
    to_currency: Optional[str] = Field(None, description="required if tool_name=convert_currency")

tool_schema = ToolStep.model_json_schema()
tool_sys_prompt = (
    "You are an orchestrator with access to tools: get_weather(location), convert_currency(amount, from_currency, to_currency). "
    "Look at the user's request and the observations so far. If any part of the request is not yet answered, "
    "set tool_name to the next tool needed and fill in its arguments. Once EVERY part has an observation, "
    "set tool_name to null. Respond with ONLY a JSON object matching this schema, no other text: " + json.dumps(tool_schema)
)

def run_orchestrator(user_request, max_steps=5):
    messages = [{"role": "system", "content": tool_sys_prompt}, {"role": "user", "content": user_request}]
    observations = []
    for step in range(max_steps):
        msg = chat(messages, response_format={"type": "json_object"}, max_tokens=250)
        parsed = ToolStep.model_validate_json(msg.content)
        print(f"[step {step}] thought={parsed.thought!r}  tool={parsed.tool_name}")
        if parsed.tool_name is None:
            break
        fn = TOOL_FUNCS.get(parsed.tool_name)
        result = fn(**parsed.model_dump()) if fn else f"error: unknown tool {parsed.tool_name}"
        print(f"          -> {result}")
        observations.append(result)
        messages.append({"role": "assistant", "content": msg.content})
        messages.append({"role": "user", "content": f"Observation: {result}"})
    return " ".join(observations) if observations else "(no tools were needed)"

print("\nFINAL:", run_orchestrator("What's the weather in Milan, and what is 50 EUR in USD?"))

Next: Resources and RAG -- data handed to the model without any tool call at all, and how to build a retrieval pipeline that decides *when* what it retrieved is actually good enough.

## Section 4 — RAG (Retrieval-Augmented Generation)

**RAG** (Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks," NeurIPS 2020, arXiv:2005.11401): combine a retriever (finds relevant text) with a generator (writes the answer using that text), so the model answers from real documents instead of only its training weights.

**The "dummy database" for this demo**, deliberately swappable: a handful of live Wikipedia summaries pulled through Wikipedia's public REST API (no key, no account). Point this at your own paper abstracts, lab notes, or documentation instead -- the rest of the pipeline (chunk -> embed -> index -> retrieve -> generate) doesn't change.

In [ ]:
import json, time, urllib.request, urllib.parse

# --- Swap this list for your own corpus (paper titles, doc names, whatever you have) ---
TOPICS = [
    "Large language model", "Retrieval-augmented generation", "Prompt engineering",
    "Natural language processing", "Model Context Protocol", "Vector database",
    "Convolutional neural network", "Reinforcement learning",
]

def fetch_corpus(topics):
    """Dummy database loader -- swap the URL/logic here for your own data source
    (a local folder of .txt files, an S3 bucket, a real paper API, etc)."""
    docs = []
    for t in topics:
        url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + urllib.parse.quote(t)
        req = urllib.request.Request(url, headers={"User-Agent": "agentic-ai-workshop-notebook/1.0"})
        try:
            with urllib.request.urlopen(req, timeout=15) as resp:
                data = json.loads(resp.read())
            docs.append({"title": t, "text": data.get("extract", "")})
        except Exception as e:
            print(f"  skipping {t!r} ({e}) -- Wikipedia rate-limits rapid requests; this is expected sometimes")
        time.sleep(0.3)  # be polite -- avoids most rate-limit errors
    return docs

corpus = fetch_corpus(TOPICS)
print(f"Loaded {len(corpus)} documents:")
for d in corpus:
    print(" -", d["title"])

In [ ]:
# --- Embed locally -- no API key, runs on CPU in a couple seconds for a corpus this size ---
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vecs = embedder.encode([d["text"] for d in corpus], normalize_embeddings=True)
print("Indexed", doc_vecs.shape[0], "documents as", doc_vecs.shape[1], "-dim vectors")

def retrieve(query, k=2):
    qvec = embedder.encode([query], normalize_embeddings=True)[0]
    sims = doc_vecs @ qvec
    top_idx = np.argsort(-sims)[:k]
    return [(corpus[i], float(sims[i])) for i in top_idx]

In [ ]:
query = "How does chain-of-thought prompting improve reasoning?"
hits = retrieve(query, k=2)
for doc, score in hits:
    print(f"  retrieved: {doc['title']} (similarity={score:.3f})")

context = "\n\n".join(f"[{d['title']}]: {d['text']}" for d, _ in hits)
answer = chat([
    {"role": "system", "content": "Answer using only the provided context. Be concise."},
    {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
])
print("\nANSWER:", answer.content)

**Where naive RAG breaks** (Singh et al., 2025, arXiv:2501.09136): it always retrieves and always answers, even when nothing in the corpus is actually relevant -- there's no loop back to try again. **Agentic RAG** adds one thing: a reflect/decide step that checks whether what it retrieved is good enough, and if not, refines the query and retrieves again. Below is a minimal version of that loop -- not the full 5-stage CRAG pipeline from the slides, just the reflect-and-retry core of it.

In [ ]:
def judge_relevance(query, hits):
    context = "\n\n".join(f"[{d['title']}]: {d['text'][:200]}" for d, _ in hits)
    msg = chat([{"role": "user", "content":
        f"Context:\n{context}\n\nDoes this context help answer: '{query}'? "
        "Reply with exactly one word: RELEVANT or IRRELEVANT."}], max_tokens=10)
    return "IRRELEVANT" not in msg.content.upper()

def refine_query(query):
    msg = chat([{"role": "user", "content":
        f"The query '{query}' did not retrieve relevant results from a corpus about AI and machine learning. "
        "Rewrite it as a short search query more likely to match documents about AI concepts. "
        "Reply with ONLY the rewritten query, nothing else."}], max_tokens=30)
    return msg.content.strip().strip('"')

def agentic_retrieve(query, k=2, max_retries=1):
    hits = retrieve(query, k)
    for attempt in range(max_retries):
        if judge_relevance(query, hits):
            return hits, query
        new_query = refine_query(query)
        print(f"  irrelevant -- refining {query!r} -> {new_query!r}")
        query = new_query
        hits = retrieve(query, k)
    return hits, query

print("Off-topic query test:")
hits, final_query = agentic_retrieve("What is the best pizza topping?")
print("Final query used:", final_query)
print("Final hits:", [d["title"] for d, _ in hits])

Next: MCP -- standardizing how tools, resources, and prompts like these get exposed, so any compliant client (not just this notebook's own hand-rolled orchestrator) can use them.

## Section 5 — MCP (Model Context Protocol)

Everything in Sections 3-4 was hand-rolled: our own tool list, our own orchestrator, our own JSON shapes. MCP standardizes that boundary so tools/resources/prompts defined once can be used by *any* compliant client, not just this notebook.

**No extra accounts or keys for this section** -- the server below is our own, running as a local subprocess talking to this notebook over stdio. (A *real* MCP server for, say, GitHub or Slack would need its own credentials; a server we write ourselves needs none.)

Note: the `mcp` Python SDK went through a breaking rename after this workshop's slides were written -- `FastMCP` became `MCPServer` in `mcp>=2.0`. We pin `mcp<2` so the code below matches `@mcp.tool()` exactly as shown in the slides.

In [ ]:
# %pip install -q "mcp<2"  # uncomment if not already installed

In [ ]:
%%writefile mcp_toy_server.py
# A tiny local MCP server -- the same trip-planning domain from the slides.
# Swap these for your own tools/resources/prompts; the client code doesn't change.
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("trip-planner-demo")

@mcp.tool()
def search_flights(origin: str, destination: str, date: str) -> str:
    """Search for flights between two cities on a given date."""
    return f"2 flights found: {origin}->{destination} on {date}: Flight AA100 ($320), Flight DL200 ($295)"

@mcp.resource("calendar://events/2024")
def calendar_2024() -> str:
    """Direct resource: fixed URI, no parameters."""
    return "2024 calendar: June 10-17 free, June 18-30 busy (conference)"

@mcp.prompt()
def plan_vacation(destination: str, duration: str, budget: str) -> str:
    """Pre-built template for planning a vacation."""
    return (
        f"Plan a {duration} trip to {destination} with a budget of {budget}. "
        "Check flights, check the calendar for free dates, and suggest an itinerary."
    )

if __name__ == "__main__":
    mcp.run()

A plain MCP client -- list what the server exposes, then call/read/get each of the three primitives:

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def explore_server():
    params = StdioServerParameters(command="python3", args=["mcp_toy_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print("TOOLS:", [t.name for t in tools.tools])
            resources = await session.list_resources()
            print("RESOURCES:", [str(r.uri) for r in resources.resources])
            prompts = await session.list_prompts()
            print("PROMPTS:", [p.name for p in prompts.prompts])

            result = await session.call_tool("search_flights", {"origin": "PHX", "destination": "BCN", "date": "2026-06-10"})
            print("\nTOOL RESULT:    ", result.content[0].text)

            res = await session.read_resource("calendar://events/2024")
            print("RESOURCE CONTENT:", res.contents[0].text)

            prompt = await session.get_prompt("plan_vacation", {"destination": "Barcelona", "duration": "7 days", "budget": "$3000"})
            print("PROMPT MESSAGE: ", prompt.messages[0].content.text)

await explore_server()  # Jupyter/Colab kernels already run an event loop, so `await` works at the top level

Now the point of the section: wire Section 3's schema-based orchestrator to call tools **through the MCP client** instead of a local Python dict. The orchestration logic (schema, loop, "trust the field not the flag") is identical -- only the execution step changes, from a direct function call to `session.call_tool(...)`. That's the whole idea of MCP: it's a standard transport for the tool-calling loop you already have, not a new agent architecture.

In [ ]:
from typing import Optional

class MCPStep(BaseModel):
    thought: str = Field(..., description="internal reasoning about what to do next")
    tool_name: Optional[str] = Field(None, description="the MCP tool to call next, or null once you have everything needed")
    origin: Optional[str] = Field(None, description="required if tool_name=search_flights")
    destination: Optional[str] = Field(None, description="required if tool_name=search_flights")
    date: Optional[str] = Field(None, description="required if tool_name=search_flights")

async def run_agent_via_mcp(user_request, max_steps=3):
    params = StdioServerParameters(command="python3", args=["mcp_toy_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            mcp_schema = MCPStep.model_json_schema()
            sys_prompt = (
                "You are an orchestrator with access to one MCP tool: search_flights(origin, destination, date). "
                "Respond with ONLY a JSON object matching this schema, no other text: " + json.dumps(mcp_schema)
            )
            messages = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_request}]
            observations = []
            for step in range(max_steps):
                msg = chat(messages, response_format={"type": "json_object"}, max_tokens=200)
                parsed = MCPStep.model_validate_json(msg.content)
                print(f"[step {step}] thought={parsed.thought!r}  tool={parsed.tool_name}")
                if parsed.tool_name is None:
                    break
                result = await session.call_tool(parsed.tool_name, {
                    "origin": parsed.origin, "destination": parsed.destination, "date": parsed.date,
                })
                text = result.content[0].text
                print(f"          -> {text}")
                observations.append(text)
                messages.append({"role": "assistant", "content": msg.content})
                messages.append({"role": "user", "content": f"Observation: {text}"})
            return " ".join(observations) if observations else "(no tools were needed)"

result = await run_agent_via_mcp("Find flights from PHX to BCN on 2026-06-10.")
print("\nFINAL:", result)

## Section 6 — Bringing It Together

Every piece above solves one problem: single-shot vs. a loop (Section 1); a loop needs a schema to know when to stop (Section 2); a schema is also the reliable way to route tool calls, since small models can't be trusted to faithfully restate a tool's result (Section 3); retrieval needs the same reflect-and-retry shape to avoid confidently answering from irrelevant context (Section 4); and MCP is a standard transport for the tool-calling loop, not a new architecture (Section 5).

The closing demo below chains three of those pieces into one request: check whether the RAG corpus actually has anything relevant (Section 4's `judge_relevance()` -- don't hand the model background that doesn't apply, it's a distraction, not a help), let a schema-driven loop decide whether it also needs to call the MCP flight-search tool (Sections 2/3/5), and -- the Section 3 lesson applied one more time -- build the final answer from the real tool observations in code rather than asking the model to restate them in prose.

In [ ]:
class FinalStep(BaseModel):
    thought: str = Field(..., description="internal reasoning about what to do next")
    tool_name: Optional[str] = Field(None, description="search_flights, or null once every needed observation has been gathered")
    origin: Optional[str] = Field(None, description="required if tool_name=search_flights")
    destination: Optional[str] = Field(None, description="required if tool_name=search_flights")
    date: Optional[str] = Field(None, description="required if tool_name=search_flights")

async def closing_demo(user_request, max_steps=4, verbose=True):
    # 1. RAG: only use retrieved background if Section 4's judge_relevance() actually says it applies.
    # (Our corpus is AI/ML topics -- for a travel question it usually won't be, and handing the model
    # irrelevant "background context" turned out, when we tested this, to be an active distraction:
    # it would get stuck reasoning about the irrelevant text instead of calling the tool.)
    hits = retrieve(user_request, k=1)
    relevant = judge_relevance(user_request, hits) if hits else False
    background = hits[0][0]["text"][:300] if (hits and relevant) else None
    background_title = hits[0][0]["title"] if (hits and relevant) else None
    if verbose:
        print("Retrieved:", hits[0][0]["title"] if hits else "(none)", "| relevant to this request:", relevant)

    # 2. MCP: same tool server as Section 5
    params = StdioServerParameters(command="python3", args=["mcp_toy_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            schema = FinalStep.model_json_schema()
            bg_line = f"Background context: {background}\n" if background else ""
            sys_prompt = (
                "You are a research travel assistant. " + bg_line +
                "You have one tool: search_flights(origin, destination, date). "
                "You MUST call search_flights before finishing whenever the user's request involves finding "
                "a flight. Set tool_name to null ONLY after you have already received a flight search "
                "observation for every leg the user asked about. "
                "Respond with ONLY a JSON object matching this schema, no other text: " + json.dumps(schema)
            )
            messages = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_request}]
            observations = []
            nudged = False
            step = 0
            while step < max_steps:
                msg = chat(messages, response_format={"type": "json_object"}, max_tokens=200)
                parsed = FinalStep.model_validate_json(msg.content)
                if verbose:
                    print(f"[step {step}] thought={parsed.thought!r}  tool={parsed.tool_name}")
                if parsed.tool_name is None:
                    if not observations and not nudged:
                        # Safety net: verified against this model -- it sometimes says "done" on step 0
                        # without ever calling the tool it just said it needed. One corrective nudge,
                        # not a silent accept, before we trust "done" here.
                        nudged = True
                        messages.append({"role": "assistant", "content": msg.content})
                        messages.append({"role": "user", "content":
                            "You have not called search_flights yet. If this request needs flight info, "
                            "set tool_name to search_flights now with the right arguments."})
                        step += 1
                        continue
                    break
                result = await session.call_tool(parsed.tool_name, {
                    "origin": parsed.origin, "destination": parsed.destination, "date": parsed.date,
                })
                text = result.content[0].text
                if verbose:
                    print(f"          -> {text}")
                observations.append(text)
                messages.append({"role": "assistant", "content": msg.content})
                messages.append({"role": "user", "content": f"Observation: {text}"})
                step += 1

            # Synthesize the final answer ourselves, from the real observations -- never ask the model
            # to restate a result in prose (Section 3's lesson, applied here one more time).
            if not observations:
                return "(no flight search was needed for this request)"
            answer = "; ".join(observations)
            if background_title:
                answer += f"  [context consulted: {background_title}]"
            return answer

answer = await closing_demo("I\'m presenting my large language model research in Barcelona -- find me a flight from PHX on 2026-06-10.")
print("\nCLOSING ANSWER:", answer)

### Where to go from here

Every placeholder in this notebook is meant to be replaced: `fetch_corpus()`'s Wikipedia calls for your own paper/document store, the mock `get_weather`/`convert_currency`/`search_flights` functions for real APIs, `BASE_URL` for a bigger model on real hardware once the shape of your agent is working here. The architecture -- one swappable OpenAI-compatible client, schema-enforced loops instead of trusting free text, tools exposed over MCP instead of hardcoded -- is the part that doesn't change as you scale up.